# Check the clisops regrid operator against CMIP6 QA/QC findings

These tests use synthetic NetCDF files to check whether clisops produces the same bilinear regridding result when projected x/y units are changed. Xarray decodes each fixture before the public clisops operator remaps it to a common target grid using xESMF and ESMPy.

| Finding | Metadata issue |
| --- | --- |
| [3.7] | Projected x/y axes with degree units |

The candidate below gives model, experiment, and variant only. It is a partial ESGF dataset identifier for future real-data validation. This notebook tests only the synthetic fixtures.

In [1]:
from importlib.metadata import version

import esmpy
import ipytest
import numpy as np
import pytest
import xarray as xr
import xesmf
from IPython.display import Markdown, display
from clisops import CONFIG
from clisops.ops.regrid import regrid

ipytest.autoconfig(raise_on_error=True, magics=False, addopts=())

for package in ("clisops", "xarray", "xesmf", "esmpy"):
    print(f"{package} {version(package)}")

clisops 0.18.1
xarray 2025.10.1
xesmf 0.9.2
esmpy 8.9.1


## Finding [3.7] — Horizontal x/y coordinate units

*Representative diagnostic*  
Projected horizontal coordinates declare `degrees` where metre units are expected.

**Reported metadata pattern.** The synthetic variant changes the units on `projection_x_coordinate` and `projection_y_coordinate` from `m` to `degrees`. The variant and metre-valued curvilinear control share valid 2-D latitude and longitude, native axis values, and data. A regular latitude/longitude grid provides a separate control for the operator and backend.

Candidate for future real-data validation — `MIROC6` / `historical` / `r1i1p1f1`.

In [2]:
LAT = {"standard_name": "latitude", "units": "degrees_north"}
LON = {"standard_name": "longitude", "units": "degrees_east"}
time = np.array(["2000-01-01", "2000-01-02"], dtype="datetime64[ns]")
lat = np.array([44.0, 45.0, 46.0, 47.0])
lon = np.array([-12.0, -11.0, -10.0, -9.0])
lon_grid, lat_grid = np.meshgrid(lon, lat)
lat_curved = lat_grid + 0.08 * (lon_grid - lon[0])
lon_curved = lon_grid + 0.06 * (lat_grid - lat[0])


def temperatures(latitude, longitude):
    return 260.0 + latitude + 2.0 * longitude + np.array([0.0, 10.0])[:, None, None]


regular = xr.Dataset(
    {"tas": (("time", "lat", "lon"), temperatures(lat_grid, lon_grid))},
    coords={"time": time, "lat": ("lat", lat, LAT), "lon": ("lon", lon, LON)},
)
curvilinear = xr.Dataset(
    {"tas": (("time", "y", "x"), temperatures(lat_curved, lon_curved))},
    coords={
        "time": time,
        "x": ("x", np.arange(4) * 80_000,
              {"standard_name": "projection_x_coordinate", "units": "m"}),
        "y": ("y", np.arange(4) * 110_000,
              {"standard_name": "projection_y_coordinate", "units": "m"}),
        "lat": (("y", "x"), lat_curved, LAT),
        "lon": (("y", "x"), lon_curved, LON),
    },
)
incorrect_units = curvilinear.copy(deep=True)
for axis in ("x", "y"):
    incorrect_units[axis].attrs["units"] = "degrees"

target = xr.Dataset(coords={
    "lat": ("lat", [44.6, 46.2], LAT),
    "lon": ("lon", [-11.4, -10.4, -9.4], LON),
})

## Pytest tests

Each run gets an empty weights cache so clisops calculates weights independently for the two curvilinear inputs. Each test checks the target grid, the two input times, and finite values. The metadata comparison depends on the valid curvilinear control and uses an absolute tolerance of `1e-10`.

The backend imports and the regular-grid test help diagnose environment or operator failures. If the curvilinear control fails, the units comparison is inconclusive. A difference with the curvilinear control passing isolates the units change in this fixture.

In [3]:
ipytest.clean()


def remap(source, directory):
    path = directory / "source.nc"
    source.to_netcdf(path, engine="netcdf4")
    with xr.open_dataset(path, engine="netcdf4") as dataset:
        decoded = dataset.load()

    for axis in ("x", "y"):
        if axis in source.coords:
            assert decoded[axis].attrs["units"] == source[axis].attrs["units"]

    weights = directory / "weights"
    weights.mkdir()
    with pytest.MonkeyPatch.context() as patch:
        patch.setitem(CONFIG["clisops:grid_weights"], "local_weights_dir", str(weights))
        outputs = regrid(decoded, grid=target.copy(deep=True), method="bilinear", output_type="xarray")
    assert len(outputs) == 1
    return outputs[0]


def assert_target(result):
    assert result.tas.dims == ("time", "lat", "lon")
    assert result.sizes == {"time": 2, "lat": 2, "lon": 3}
    np.testing.assert_array_equal(result.time, time)
    np.testing.assert_array_equal(result.lat, target.lat)
    np.testing.assert_array_equal(result.lon, target.lon)
    assert np.isfinite(result.tas.values).all()


@pytest.fixture(scope="module", params=[pytest.param(curvilinear, id="3.7-curvilinear-control")])
def curvilinear_control(request, tmp_path_factory):
    return remap(request.param, tmp_path_factory.mktemp("curvilinear-control"))


@pytest.mark.parametrize("source", [pytest.param(regular, id="3.7-regular-control")])
def test_regular_grid(source, tmp_path):
    assert_target(remap(source, tmp_path))


def test_curvilinear_grid(curvilinear_control):
    assert_target(curvilinear_control)


@pytest.mark.parametrize("source", [pytest.param(incorrect_units, id="3.7-degree-units")])
def test_projected_axis_units(curvilinear_control, source, tmp_path):
    actual = remap(source, tmp_path)
    assert_target(actual)
    np.testing.assert_allclose(actual.tas, curvilinear_control.tas, rtol=0, atol=1e-10)
    difference = np.abs(actual.tas.values - curvilinear_control.tas.values).max()
    print(f"Maximum absolute difference from the curvilinear control: {difference:.3g}")

## Run pytest

In [4]:
FINDINGS = {"3.7": "Horizontal x/y coordinate units"}


class FindingResults:
    def __init__(self):
        self.tests = {finding: [] for finding in FINDINGS}
        self.outcomes = {}

    def pytest_itemcollected(self, item):
        # A shared control can contribute to more than one finding.
        for finding in item.callspec.id.split("-", 1)[0].split("+"):
            self.tests[finding].append(item.nodeid)

    def pytest_runtest_logreport(self, report):
        phase_outcomes = self.outcomes.setdefault(report.nodeid, {})
        phase_outcomes[report.when] = "skipped" if hasattr(report, "wasxfail") else report.outcome

    def status(self, finding):
        outcomes = [self.outcomes.get(nodeid, {}) for nodeid in self.tests[finding]]
        passed = {"setup": "passed", "call": "passed", "teardown": "passed"}
        if any("failed" in phases.values() for phases in outcomes):
            return "FAIL"
        if outcomes and all(phases == passed for phases in outcomes):
            return "PASS"
        return "INCOMPLETE"


results = FindingResults()
pytest_exit = ipytest.run(
    "-v", "-rP", "--color=no", "-p", "no:cacheprovider",
    plugins=[results], raise_on_error=False,
)

======================================= test session starts ========================================
platform darwin -- Python 3.13.18, pytest-9.2.0, pluggy-1.7.0 -- /Users/pingu/miniforge3/envs/c3s-cmip6-update-tests/bin/python
rootdir: /Users/pingu/Documents/GitHub/macpingu/c3s-cmip6-update-tests/notebooks
plugins: platformdirs-4.14.5, anyio-4.16.0
collecting ... collected 3 items

t_5df731a36215450fb9f7a73bef2fa224.py::test_regular_grid[3.7-regular-control] PASSED         [ 33%]
t_5df731a36215450fb9f7a73bef2fa224.py::test_curvilinear_grid[3.7-curvilinear-control] PASSED [ 66%]
t_5df731a36215450fb9f7a73bef2fa224.py::test_projected_axis_units[3.7-curvilinear-control-3.7-degree-units] PASSED [100%]

========================================= warnings summary =========================================
t_5df731a36215450fb9f7a73bef2fa224.py: 12 warnings
  /Users/pingu/miniforge3/envs/c3s-cmip6-update-tests/lib/python3.13/site-packages/clisops/utils/dataset_utils.py:1853: UserWarning: For co

## Results and limitations

A PASS requires both controls and the degree-units comparison to pass. Pytest shows the maximum absolute difference and any warnings. The synthetic grids omit cell bounds, so clisops can warn about missing latitude or longitude bounds. The Woodpecker assessment applies to these fixtures, which run without preprocessing.

This run covers bilinear regridding on small single-file inputs with valid geographic coordinates. Other methods, grids, backend versions, and real datasets may behave differently.

In [5]:
table = [
    "| Finding | Description | Regridding result | Woodpecker required? (synthetic) |",
    "| --- | --- | --- | --- |",
]
for finding, description in FINDINGS.items():
    status = results.status(finding)
    woodpecker = "Not needed" if status == "PASS" else "To investigate"
    table.append(f"| [{finding}] | {description} | {status} | {woodpecker} |")

display(Markdown("\n".join(table)))
assert pytest_exit == pytest.ExitCode.OK, f"pytest exited with {pytest_exit.name}"

| Finding | Description | Regridding result | Woodpecker required? (synthetic) |
| --- | --- | --- | --- |
| [3.7] | Horizontal x/y coordinate units | PASS | Not needed |